# LAB 07 — Transformer Classification Experiments

Notebook này triển khai Bài 19–23, 25 và 29 theo W7.pdf. Mọi metrics/đồ thị được tạo từ model thực sự chạy. Các phần nhận xét cá nhân để sinh viên tự hoàn thiện.

## 1. Imports & Environment

Import TransformerBlock tự cài ở Bài 17; chạy regression tests để chắc thay đổi mask/inspection vẫn giữ hoạt động cũ.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
from IPython.display import display

LAB_DIR = Path.cwd()
if not (LAB_DIR / "W7.pdf").exists():
    LAB_DIR = (LAB_DIR / "Lab7").resolve()
sys.path.insert(0, str(LAB_DIR))

from transformer_block import TransformerBlock, run_tests as test_transformer_block
from transformer_experiments import (
    ExperimentRunner, SEED, SPLIT_SEED, D_MODEL, D_FF, BATCH_SIZE,
    LEARNING_RATE, EPOCHS, MAX_LENGTH, MAX_LENGTHS,
)

test_transformer_block()

/home/nguyenducthang/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Bài 17 — Unit tests
Bài 18 — Tensor shapes thực tế
Input X: (8, 100, 128)
Q before splitting heads: (8, 100, 128)
K before splitting heads: (8, 100, 128)
V before splitting heads: (8, 100, 128)
Q after splitting heads: (8, 4, 100, 32)
K after splitting heads: (8, 4, 100, 32)
V after splitting heads: (8, 4, 100, 32)
Attention scores: (8, 4, 100, 100)
Attention weights: (8, 4, 100, 100)
Attention output per head: (8, 4, 100, 32)
Concatenated heads: (8, 100, 128)
Output projection: (8, 100, 128)
Residual + LayerNorm 1: (8, 100, 128)
FFN hidden: (8, 100, 512)
FFN output: (8, 100, 128)
Residual + LayerNorm 2: (8, 100, 128)
Final output: (8, 100, 128)
Attention weight row-sum max error: 2.384e-07
Attention weights sum to one: True
PASS: input/output và intermediate shapes theo cấu hình đề bài
PASS: Q/K/V tách head, scores và weights đúng shapes
PASS: attention weights cộng bằng 1 trên chiều key
PASS: hai residual connections đưa đúng tensor vào LayerNorm
PASS: padding keys bị mask và API tr

## 2. Configuration & Reproducibility

`full` dùng toàn bộ split LAB5 và 5 epochs. `smoke` chỉ kiểm tra pipeline trên các tập con xác định (2.048 train, 512 validation, 1.024 test) với 1 epoch; không dùng smoke metrics thay cho kết quả full.

In [2]:
RUN_MODE = "full"  # đổi thành "smoke" để kiểm tra nhanh pipeline
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "seed": SEED,
    "split_seed_lab5": SPLIT_SEED,
    "d_model": D_MODEL,
    "d_ff": D_FF,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS if RUN_MODE == "full" else 1,
    "max_length_default": MAX_LENGTH,
    "max_lengths": list(MAX_LENGTHS),
    "transformer_blocks": 1,
    "run_mode": RUN_MODE,
    "device": DEVICE,
}
print(CONFIG)
print("CUDA available:", torch.cuda.is_available())

{'seed': 42, 'split_seed_lab5': 2026, 'd_model': 128, 'd_ff': 512, 'batch_size': 32, 'learning_rate': 0.001, 'epochs': 5, 'max_length_default': 128, 'max_lengths': [32, 64, 128, 256], 'transformer_blocks': 1, 'run_mode': 'full', 'device': 'cpu'}
CUDA available: False


## 3. Dataset Loading — IMDb, đúng split LAB5

Dùng split IMDb có nhãn `train` và `test`; không dùng `unsupervised`. IMDb train được chia 80/20 stratified với `random_state=2026` như LAB5; test gốc giữ nguyên. Model seed là 42.

In [3]:
runner = ExperimentRunner(LAB_DIR, device=DEVICE, run_mode=RUN_MODE)
ENVIRONMENT = runner.environment_summary()
display(pd.Series(ENVIRONMENT, name="Value").to_frame())
print("Train/validation/test sizes:", len(runner.train_ids), len(runner.val_ids), len(runner.test_labels))
print("Vocabulary size, including PAD/UNK:", len(runner.vocab))
print("Label mapping: 0=Negative, 1=Positive; positive class for F1=1")

,Value
torch,2.12.1+cpu
device,cpu
cuda_available,False
dataset,IMDb labeled train/test; unsupervised excluded
split,"train=20000, validation=5000, test=25000"
split_seed,2026
model_seed,42
vocabulary_size_including_PAD_UNK,20000
batch_size,32
epochs,5


Train/validation/test sizes: 20000 5000 25000
Vocabulary size, including PAD/UNK: 20000
Label mapping: 0=Negative, 1=Positive; positive class for F1=1


## 4. Tokenizer & Vocabulary

Tái sử dụng cách làm sạch/tokenizer regex LAB5. Vocabulary tối đa 20.000 mục, min frequency 2, chỉ dựng từ train. PAD và UNK có ID riêng. Review được truncate bên phải theo max length và padding động theo batch.

In [4]:
print("Tokenizer: HTML unescape + lowercase + strip HTML + [a-z0-9]+(?:'[a-z]+)?")
print("PAD_ID:", 0, "UNK_ID:", 1)
print("Vocabulary sample:", list(runner.vocab.items())[:12])
for split, ids, labels in (
    ("train", runner.train_ids, runner.train_labels),
    ("validation", runner.val_ids, runner.val_labels),
):
    print(split, "samples:", len(ids), "labels:", pd.Series(labels).value_counts().sort_index().to_dict())

Tokenizer: HTML unescape + lowercase + strip HTML + [a-z0-9]+(?:'[a-z]+)?
PAD_ID: 0 UNK_ID: 1
Vocabulary sample: [('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('and', 3), ('a', 4), ('of', 5), ('to', 6), ('is', 7), ('in', 8), ('it', 9), ('i', 10), ('this', 11)]
train samples: 20000 labels: {0: 10000, 1: 10000}
validation samples: 5000 labels: {0: 2500, 1: 2500}


## 5. Shared Training & Evaluation Utilities

`ExperimentRunner` dùng chung loaders, optimizer Adam, CrossEntropyLoss, metrics binary (positive=1), early checkpoint selection theo Validation F1 và loss trung bình theo số samples. Test chỉ được gọi ở phần final evaluation.

In [5]:
print("Batch size:", BATCH_SIZE, "| LR:", LEARNING_RATE, "| epochs:", CONFIG["epochs"])
print("Attention masks exclude PAD keys; pooling excludes PAD positions.")
print("Train shuffling restarts with the same seed for each independent model.")

Batch size: 32 | LR: 0.001 | epochs: 5
Attention masks exclude PAD keys; pooling excludes PAD positions.
Train shuffling restarts with the same seed for each independent model.


## 6. Transformer Classifier — Bài 23

Pipeline: Embedding → sinusoidal positional encoding → một TransformerBlock tự cài → masked mean pooling → Linear classifier. Classifier trả logits; xác suất chỉ tính khi cần prediction. Có thể bật learned `[CLS]` vector ở Bài 25.

In [6]:
print("TransformerBlock source:", (LAB_DIR / "transformer_block.py").resolve())
print("Padding mask is applied to attention keys and to mean pooling.")
print("One block; d_model=128, d_ff=512, dropout=0.1.")

TransformerBlock source: /home/nguyenducthang/NLP/Lab7/transformer_block.py
Padding mask is applied to attention keys and to mean pooling.
One block; d_model=128, d_ff=512, dropout=0.1.


## 7. Baseline Classification & 8. Number of Attention Heads — Bài 23 và 19

Mỗi số head tạo model mới, seed/optimizer/data/batch/epochs/length/pooling giữ nguyên. Bảng dùng Validation metrics để so sánh và chọn checkpoint; cột parameters là toàn classifier và có thêm số riêng của TransformerBlock.

In [7]:
heads_results = runner.run_heads()
display(heads_results.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}", "Best validation loss": "{:.4f}",
}))

heads_1 | epoch 1/5 | train_loss=0.5664 | val_loss=0.4759 | val_f1=0.7403
heads_1 | epoch 2/5 | train_loss=0.3916 | val_loss=0.3852 | val_f1=0.8308
heads_1 | epoch 3/5 | train_loss=0.2928 | val_loss=0.4109 | val_f1=0.8354
heads_1 | epoch 4/5 | train_loss=0.2171 | val_loss=0.4078 | val_f1=0.8361
heads_1 | epoch 5/5 | train_loss=0.1479 | val_loss=0.5109 | val_f1=0.8285
heads_1: best validation epoch=4, training+validation time=214.80s
heads_2 | epoch 1/5 | train_loss=0.5467 | val_loss=0.4568 | val_f1=0.7588
heads_2 | epoch 2/5 | train_loss=0.3788 | val_loss=0.4019 | val_f1=0.8299
heads_2 | epoch 3/5 | train_loss=0.2815 | val_loss=0.3968 | val_f1=0.8374
heads_2 | epoch 4/5 | train_loss=0.2039 | val_loss=0.4126 | val_f1=0.8384
heads_2 | epoch 5/5 | train_loss=0.1288 | val_loss=0.5204 | val_f1=0.8268
heads_2: best validation epoch=4, training+validation time=155.39s
heads_4 | epoch 1/5 | train_loss=0.5482 | val_loss=0.4636 | val_f1=0.7554
heads_4 | epoch 2/5 | train_loss=0.3814 | val_loss=0

,Heads,d_k,Classifier parameters,TransformerBlock parameters,Training time (s),Validation accuracy,Validation F1,Best validation loss
0,1,128,2758530,198272,214.80,0.8370,0.8361,0.4078
1,2,64,2758530,198272,155.39,0.8378,0.8384,0.4126
2,4,32,2758530,198272,206.40,0.8300,0.8333,0.5323
3,8,16,2758530,198272,213.58,0.8250,0.8339,0.3882


## 9. Attention Head Diversity — Bài 20

Lấy attention weights từ model 4-head đã train, ở `eval()` mode. Heatmaps dùng chung scale; thống kê entropy, max weight và cosine similarity chỉ mô tả phân phối, không tự chứng minh ý nghĩa ngôn ngữ.

In [8]:
head_stats, head_similarity = runner.run_head_diversity()
display(head_stats.style.format({
    "Mean attention entropy": "{:.4f}", "Mean maximum weight": "{:.4f}"
}))
display(head_similarity.style.format("{:.4f}"))

Attention sentence: i wouldn't rent this one even on dollar rental night
Saved: /home/nguyenducthang/NLP/Lab7/attention_heads.png


,Head,Mean attention entropy,Mean maximum weight
0,1,1.4856,0.4776
1,2,0.9608,0.6946
2,3,1.1002,0.6189
3,4,1.0169,0.6398


,Head 1,Head 2,Head 3,Head 4
Head 1,1.0000,0.5926,0.1380,0.1767
Head 2,0.5926,1.0000,0.1039,0.1957
Head 3,0.1380,0.1039,1.0000,0.2402
Head 4,0.1767,0.1957,0.2402,1.0000


## 10. Transformer vs LSTM — Bài 21

LSTM dùng packed sequences để bỏ padding; hai model dùng chung data split, vocabulary, embedding/hidden size 128, batch, optimizer, LR, epochs và max length 128. Số parameters không ép bằng nhau. Bảng dưới là Validation; test sẽ được báo cáo một lần ở phần cuối.

In [9]:
comparison_validation = runner.run_lstm()
display(comparison_validation.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}", "Best validation loss": "{:.4f}",
}))

lstm | epoch 1/5 | train_loss=0.6866 | val_loss=0.6513 | val_f1=0.5904
lstm | epoch 2/5 | train_loss=0.6179 | val_loss=0.5257 | val_f1=0.7551
lstm | epoch 3/5 | train_loss=0.4350 | val_loss=0.4203 | val_f1=0.8194
lstm | epoch 4/5 | train_loss=0.3094 | val_loss=0.3958 | val_f1=0.8347
lstm | epoch 5/5 | train_loss=0.2274 | val_loss=0.4111 | val_f1=0.8342
lstm: best validation epoch=4, training+validation time=298.20s


,Model,Parameters,Training time (s),Validation accuracy,Validation F1,Best validation loss
0,"Transformer (4 heads, mean)",2758530,206.40,0.8300,0.8333,0.5323
1,LSTM,2692354,298.20,0.8298,0.8347,0.3958


## 11. Sequence Length — Bài 22

So sánh max_length 32/64/128/256. Length 128 tái sử dụng đúng model 4-head đã train ở Bài 19. Tokens retained được đo trên train; attention memory là ước lượng cho **một** score/weight matrix `(batch, heads, T, T)`, không phải tổng bộ nhớ training.

In [10]:
length_results = runner.run_sequence_lengths()
display(length_results.style.format({
    "Mean tokens retained/sample": "{:.2f}", "Train tokens retained (%)": "{:.2f}%",
    "Training time (s)": "{:.2f}", "Validation F1": "{:.4f}",
    "Validation accuracy": "{:.4f}",
}))

length_32 | epoch 1/5 | train_loss=0.6275 | val_loss=0.5701 | val_f1=0.7204
length_32 | epoch 2/5 | train_loss=0.5355 | val_loss=0.5286 | val_f1=0.7406
length_32 | epoch 3/5 | train_loss=0.4548 | val_loss=0.5251 | val_f1=0.7447
length_32 | epoch 4/5 | train_loss=0.3735 | val_loss=0.5695 | val_f1=0.7218
length_32 | epoch 5/5 | train_loss=0.2908 | val_loss=0.6644 | val_f1=0.7390
length_32: best validation epoch=3, training+validation time=55.44s
length_64 | epoch 1/5 | train_loss=0.5999 | val_loss=0.5198 | val_f1=0.7600
length_64 | epoch 2/5 | train_loss=0.4638 | val_loss=0.4522 | val_f1=0.7890
length_64 | epoch 3/5 | train_loss=0.3673 | val_loss=0.4622 | val_f1=0.7992
length_64 | epoch 4/5 | train_loss=0.2825 | val_loss=0.5043 | val_f1=0.7864
length_64 | epoch 5/5 | train_loss=0.1984 | val_loss=0.6150 | val_f1=0.7911
length_64: best validation epoch=3, training+validation time=76.95s
length_256 | epoch 1/5 | train_loss=0.5078 | val_loss=0.3991 | val_f1=0.8032
length_256 | epoch 2/5 | tr

,Length,Mean tokens retained/sample,Train tokens retained (%),One attention matrix (bytes),Training time (s),Validation F1,Validation accuracy
0,32,31.98,13.62%,524288,55.44,0.7447,0.7342
1,64,63.31,26.97%,2097152,76.95,0.7992,0.7906
2,128,119.47,50.89%,8388608,206.40,0.8333,0.8300
3,256,179.46,76.44%,33554432,462.90,0.8654,0.8642


## 12. Pooling Strategy — Bài 25

Mean pooling loại padding. CLS-style dùng một learned `[CLS]` vector đặt ở đầu sequence; positional encoding/mask được dịch và cập nhật tương ứng. Chỉ thay pooling và token cần thiết.

In [11]:
pooling_results = runner.run_pooling()
display(pooling_results.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}",
}))

pooling_cls | epoch 1/5 | train_loss=0.5515 | val_loss=0.4591 | val_f1=0.7673
pooling_cls | epoch 2/5 | train_loss=0.4107 | val_loss=0.4335 | val_f1=0.8179
pooling_cls | epoch 3/5 | train_loss=0.3196 | val_loss=0.3856 | val_f1=0.8359
pooling_cls | epoch 4/5 | train_loss=0.2549 | val_loss=0.4153 | val_f1=0.8284
pooling_cls | epoch 5/5 | train_loss=0.1949 | val_loss=0.4414 | val_f1=0.8289
pooling_cls: best validation epoch=3, training+validation time=158.48s


,Pooling,Parameters,Validation accuracy,Validation F1,Training time (s)
0,Mean,2758530,0.8300,0.8333,206.40
1,CLS-style (learned token),2758658,0.8334,0.8359,158.48


## 13. Learning Curves — Bài 29

Loss được lưu từng epoch, train loss theo train samples và validation loss theo validation samples. Biểu đồ không tự gán kết luận overfitting/underfitting.

In [12]:
learning_history = runner.save_learning_curves()
display(learning_history.style.format({
    "train_loss": "{:.4f}", "validation_loss": "{:.4f}", "validation_f1": "{:.4f}"
}))

Saved: /home/nguyenducthang/NLP/Lab7/learning_curves.png /home/nguyenducthang/NLP/Lab7/learning_curves.csv


,experiment,epoch,train_loss,validation_loss,validation_f1
0,heads_4,1,0.5482,0.4636,0.7554
1,heads_4,2,0.3814,0.4155,0.8261
2,heads_4,3,0.2857,0.3849,0.8302
3,heads_4,4,0.2033,0.4348,0.8287
4,heads_4,5,0.1281,0.5323,0.8333
5,lstm,1,0.6866,0.6513,0.5904
6,lstm,2,0.6179,0.5257,0.7551
7,lstm,3,0.4350,0.4203,0.8194
8,lstm,4,0.3094,0.3958,0.8347
9,lstm,5,0.2274,0.4111,0.8342


## 14. Confusion Matrix & Final Test Evaluation

Sau khi cấu hình chính được chọn qua Validation, đánh giá Transformer 4-head mean baseline và LSTM trên IMDb test split gốc (hoặc test subset nếu RUN_MODE=smoke). Confusion matrix tương ứng Transformer; tổng ô phải bằng số test samples.

In [13]:
test_results, test_labels, transformer_test_predictions = runner.final_test_evaluation()
display(test_results.style.format({
    "Training time (s)": "{:.2f}", "Accuracy": "{:.4f}",
    "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}",
}))
print("Evaluated test samples:", len(test_labels))

,Model,Parameters,Training time (s),Accuracy,Precision,Recall,F1
0,"Transformer (4 heads, mean)",2758530,206.40,0.8071,0.8029,0.8142,0.8085
1,LSTM,2692354,298.20,0.8161,0.8065,0.8318,0.8189


Evaluated test samples: 25000


## 15. Application — Năm loại input

Đây là kiểm tra hành vi trên câu thủ công, không thay evaluation test. Xác suất lấy softmax từ logits của model đã train.

In [14]:
example_predictions = runner.predict_examples()
display(example_predictions.style.format({"P(Negative)": "{:.4f}", "P(Positive)": "{:.4f}"}))

,Type,Input text,Predicted label,P(Negative),P(Positive),Original tokens,Retained tokens
0,Short,I loved this movie.,Positive,0.0000,1.0000,4,4
1,Long,"The opening was promising and the actors were convincing, but the plot became repetitive, the middle dragged, and the ending was disappointing despite a few beautiful scenes and one excellent performance.",Negative,0.9990,0.0010,31,31
2,Negation,I did not enjoy this film.,Positive,0.0215,0.9785,6,6
3,Contrast,This movie has a good beginning but the ending is terrible.,Negative,0.9930,0.0070,11,11
4,Ambiguous,"The movie was not bad, although it was not especially memorable.",Negative,0.9981,0.0019,11,11


## 16. Export Results

`results.csv` có các dòng riêng cho từng experiment, validation/test split và epoch loss. Trường không áp dụng để trống; kết quả smoke được đánh dấu `run_mode=smoke`.

In [15]:
results = runner.export_results()
display(results.groupby(["experiment", "eval_split"], dropna=False).size().rename("rows").to_frame())
print("Results path:", LAB_DIR / "results.csv")
print("Images:", LAB_DIR / "attention_heads.png", LAB_DIR / "learning_curves.png", LAB_DIR / "confusion_matrix.png")

Saved 28 measured rows to /home/nguyenducthang/NLP/Lab7/results.csv


,,rows
experiment,eval_split,
Bài 19 — Attention heads,validation,4
Bài 20 — Attention diversity,validation,4
Bài 21 — LSTM vs Transformer,validation,2
Bài 22 — Sequence length,validation,4
Bài 25 — Pooling strategy,validation,2
Bài 29 — Learning curves,validation,10
Final test evaluation,test,2


Results path: /home/nguyenducthang/NLP/Lab7/results.csv
Images: /home/nguyenducthang/NLP/Lab7/attention_heads.png /home/nguyenducthang/NLP/Lab7/learning_curves.png /home/nguyenducthang/NLP/Lab7/confusion_matrix.png


Dưới đây là phần trả lời 6 câu hỏi thực nghiệm LAB 07, dựa trên số liệu thực tế trong `results.csv` và `learning_curves.csv` mà bạn đã chạy và push lên GitHub. Các nhận xét được viết ngắn gọn, phù hợp để tham khảo khi hoàn thiện notebook `transformer_classification.ipynb`.

## Bài 19. Number of Attention Heads

Câu hỏi: Tăng số heads có tăng representation dimension không? \\(d_k\\) thay đổi ra sao? Vì sao F1 có thể không tăng?

Trả lời:

Tăng số attention heads không đồng nghĩa với tăng representation dimension khi \\(d\_{model}\\) được giữ cố định.

Theo công thức:

\\[ d_k=\frac{d\_{model}}{n\_{heads}} \\]

Với \\(d\_{model}=128\\):

| Heads | \\(d_k\\) | Accuracy | F1     |
| ----- | --------- | -------- | ------ |
| 1     | 128       | 0.8370   | 0.8361 |
| 2     | 64        | 0.8378   | 0.8384 |
| 4     | 32        | 0.8300   | 0.8333 |
| 8     | 16        | 0.8250   | 0.8339 |

Kết quả cho thấy mô hình 2 heads đạt F1 cao nhất (0.8384). Khi tiếp tục tăng lên 4 hoặc 8 heads, F1 không tăng tương ứng.

Nguyên nhân là khi \\(d\_{model}\\) cố định, số chiều của mỗi head giảm khi tăng số heads. Nhiều heads có thể học các attention patterns đa dạng hơn, nhưng không đảm bảo cải thiện khả năng phân loại. Trong implementation này, số parameters của toàn classifier vẫn là 2.758.530 ở cả bốn cấu hình.

Kết luận: Tăng số heads không chắc chắn cải thiện F1; hiệu quả phụ thuộc vào cách mô hình học và phân bổ thông tin giữa các heads.

## Bài 20. Attention Head Diversity

Câu hỏi: Head nào tập trung hoặc phân tán? Các attention matrices có giống nhau không? Evidence nào hỗ trợ nhận xét?

Trả lời:

Dựa trên kết quả đo attention entropy và mean maximum attention weight:

| Head   | Mean entropy | Mean max weight |
| ------ | ------------ | --------------- |
| Head 1 | 1.4856       | 0.4776          |
| Head 2 | 0.9608       | 0.6946          |
| Head 3 | 1.1002       | 0.6189          |
| Head 4 | 1.0169       | 0.6398          |

- Head 2 tập trung mạnh nhất, vì có entropy thấp nhất và maximum attention weight trung bình cao nhất.
- Head 1 phân tán nhất, vì có entropy cao nhất và maximum attention weight trung bình thấp nhất.
- Head 3 và Head 4 có mức độ tập trung nằm giữa hai head trên.

Các attention patterns không hoàn toàn giống nhau. Chẳng hạn, Head 3 có mean pairwise cosine similarity thấp nhất, khoảng 0.1607, cho thấy pattern của nó tương đối khác những heads còn lại trong mẫu được kiểm tra.

Kết luận: Multi-head attention đã tạo ra các phân phối attention khác nhau trong thí nghiệm này. Tuy nhiên, các số liệu chỉ được đo trên một review validation, nên chưa thể kết luận mỗi head học một chức năng ngôn ngữ riêng hoặc chứng minh head nào dư thừa.

## Bài 21. Transformer vs LSTM

Câu hỏi: Khác biệt kiến trúc nào có thể liên quan đến kết quả? Sequential và parallel processing khác gì?

Trả lời:

Kết quả trên validation set:

| Model       | Parameters | Training time | F1     |
| ----------- | ---------- | ------------- | ------ |
| LSTM        | 2.692.354  | 298.20 s      | 0.8347 |
| Transformer | 2.758.530  | 206.40 s      | 0.8333 |

LSTM xử lý sequence theo từng timestep, trong đó hidden state hiện tại phụ thuộc vào các hidden states trước đó. Ngược lại, Transformer sử dụng self-attention để tính toán quan hệ giữa các token một cách song song hơn.

Trong thí nghiệm, Transformer có thời gian training thấp hơn LSTM, nhưng F1 validation lại thấp hơn một chút.

Điều này có thể liên quan đến cách hai kiến trúc biểu diễn thông tin: LSTM sử dụng recurrent hidden states, còn Transformer sử dụng positional encoding, self-attention và mean pooling. Ngoài ra, số parameters cũng không hoàn toàn bằng nhau.

Trên test set, F1 của LSTM đạt 0.8189, cao hơn Transformer (0.8085).

Kết luận: Transformer thể hiện lợi thế về thời gian chạy trong cấu hình đã thử, nhưng không đạt hiệu quả phân loại cao hơn LSTM. Tốc độ và độ chính xác phụ thuộc vào kiến trúc, dữ liệu và phần cứng.

## Bài 22. Sequence Length

Câu hỏi: Vì sao không chỉ tăng sequence length? \\(T^2\\) và thời gian thực tế thay đổi ra sao?

Trả lời:

Self-attention có ma trận attention kích thước \\(T\times T\\), nên số phần tử tăng theo:

\\[ O(T^2) \\]

Kết quả thực nghiệm:

| Length | Training time | F1     |
| ------ | ------------- | ------ |
| 32     | 55.44 s       | 0.7447 |
| 64     | 76.95 s       | 0.7992 |
| 128    | 206.40 s      | 0.8333 |
| 256    | 462.90 s      | 0.8654 |

Khi sequence length tăng từ 128 lên 256:

\\[ \frac{256^2}{128^2}=4 \\]

Tức là số phần tử trong attention matrix tăng 4 lần. Bộ nhớ lý thuyết cho một attention matrix tăng từ 8 MiB lên 32 MiB, trong khi thời gian huấn luyện thực tế tăng khoảng 2,24 lần.

Nếu tăng từ 32 lên 256, số phần tử attention matrix tăng 64 lần, nhưng thời gian thực tế tăng khoảng 8,35 lần.

F1 có xu hướng cải thiện khi sequence length tăng vì mô hình giữ được nhiều token hơn, hạn chế mất thông tin do truncation.

Kết luận: Không thể tăng sequence length vô hạn vì chi phí tính toán và bộ nhớ tăng nhanh. Cần cân bằng giữa lượng thông tin giữ lại, hiệu quả phân loại và tài nguyên phần cứng. Thời gian chạy thực tế không nhất thiết tăng đúng theo tỷ lệ \\(T^2\\).

## Bài 25. Pooling Strategy

Câu hỏi: Pooling ảnh hưởng gì tới representation và kết quả trên split này?

Trả lời:

Pooling chuyển representation của các token thành một vector đại diện cho toàn bộ sequence để thực hiện classification.

- Mean pooling: Lấy trung bình các token representations, loại bỏ padding. Phương pháp này tổng hợp thông tin từ toàn bộ sequence.
- CLS-style pooling: Sử dụng representation của một learned CLS token đặt ở đầu sequence để thực hiện classification.

Kết quả validation:

| Pooling   | Accuracy | F1     | Training time |
| --------- | -------- | ------ | ------------- |
| Mean      | 0.8300   | 0.8333 | 206.40 s      |
| CLS-style | 0.8334   | 0.8359 | 158.48 s      |

CLS-style pooling có F1 cao hơn Mean pooling khoảng 0.0027, đồng thời có thời gian training thấp hơn trong lần chạy này.

Sự khác biệt cho thấy cách tổng hợp token representations có thể ảnh hưởng đến khả năng phân loại. Tuy nhiên, mức chênh lệch F1 khá nhỏ và chỉ được đo với một seed, nên chưa đủ để khẳng định CLS-style luôn tốt hơn.

Kết luận: Pooling strategy ảnh hưởng đến sequence representation và performance. Trong thí nghiệm này, CLS-style cho kết quả nhỉnh hơn một chút.

## Bài 29. Learning Curves

Câu hỏi: Quan sát convergence, instability và overfitting từ từng đường loss.

Trả lời:

### Transformer

Transformer learning curves

Training lossValidation loss

00,150,30,450,612345

Training loss giảm liên tục từ 0.5482 xuống 0.1281, cho thấy mô hình đang học trên training set.

Validation loss giảm đến epoch 3, đạt 0.3849, sau đó tăng lên 0.5323 ở epoch 5. Đây là dấu hiệu overfitting theo loss: mô hình tiếp tục cải thiện trên train nhưng validation loss trở nên xấu hơn.

Tuy nhiên, Validation F1 ở epoch 5 (0.8333) vẫn nhỉnh hơn epoch 3 (0.8302). Do đó, checkpoint tốt nhất theo F1 không nhất thiết trùng với checkpoint tốt nhất theo validation loss.

### LSTM

LSTM learning curves

Training lossValidation loss

0,150,30,450,60,7512345

Training loss của LSTM giảm từ 0.6866 xuống 0.2274. Validation loss cũng giảm đến epoch 4, đạt giá trị thấp nhất 0.3958, rồi tăng nhẹ ở epoch 5.

Điều này cho thấy LSTM có xu hướng hội tụ trong những epoch đầu và bắt đầu xuất hiện dấu hiệu overfitting theo validation loss ở epoch cuối.

Kết luận chung: Cả hai mô hình đều có training loss giảm, nhưng validation loss không giảm liên tục. Transformer xuất hiện dấu hiệu overfitting sau epoch 3, còn LSTM sau epoch 4. Chưa có bằng chứng đủ mạnh để kết luận training bị mất ổn định nghiêm trọng hoặc bị underfitting.

